### Appendix. Remote Sensing and Machine Learning-Based Tungsten Prospectivity Mapping in Primorsky Krai, Russian Far East
**Sergei Shevyrev, Irina Tarasenko, and Natalia Boriskina**

*FEGI FEB RAS, Vladivostok, Russia*

### Processing of topographically corrected Landsat 8 OLI bands to compute mineral components
The algorithm uses directed principal components method. It selects all possible two band ratios using Python for every mineral 
("quartz","alunite", "illite","chlorite","goethite","epidote") and noise signal of “vegetation” value based on a rule: mineral and vegetation band ratios  must be greater that 1 for vegetatoin reflectances, when for mineral they can be greater or lower than 1.
Possibly "mineral" and "vegetation" components are saved into filenames stored in MINERAL_RATIOS and VEGETATION_RATIOS.

Reflectance values for "minerals" and "vegetations" are taken from [1,2].

***References***
<ol>
<li>R.N. Clark, G.A. Swayze, A. Gallagher, T.V.V. King, W.M. Calvin The US Geological Survey Digital Spectral Library, Version 1:0.2 to 3.0 mm US Geological Survey Open File Report (1993), pp. 93-592 </li>
<li>Hubbard, B. E., Crowley, J. K., Zimbelman D.R., 2003 Comparative alteration mineral mapping using visible to shortwave infrared (0.4–2.4μm) Hyperion, ALI and ASTER Imagery. IEEE Trans. Geosci. Remote Sens., 41 1401 –1410 (2003). doi: 10.1109/TGRS.2003.812906IGRSD20196-2892.</li>
</ol>

In [ ]:
"""
{
  "Reflectance_spectra": {
    "band1": {
      "quartz": 0.72,
      "alunite": 0.69,
      "illite": 0.26,
      "chlorite": 0.17,
      "goethite": 0.07,
      "epidote": 0.45,
      "vegetation": 0.04
    },
    "band2": {
      "quartz": 0.74,
      "alunite": 0.73,
      "illite": 0.31,
      "chlorite": 0.21,
      "goethite": 0.07,
      "epidote": 0.07,
      "vegetation": 0.05
    },
    "band3": {
      "quartz": 0.77,
      "alunite": 0.81,
      "illite": 0.38,
      "chlorite": 0.22,
      "goethite": 0.17,
      "epidote": 0.12,
      "vegetation": 0.09
    },
    "band4": {
      "quartz": 0.78,
      "alunite": 0.82,
      "illite": 0.34,
      "chlorite": 0.18,
      "goethite": 0.18,
      "epidote": 0.16,
      "vegetation": 0.06
    },
    "band5": {
      "quartz": 0.79,
      "alunite": 0.83,
      "illite": 0.31,
      "chlorite": 0.18,
      "goethite": 0.19,
      "epidote": 0.26,
      "vegetation": 0.55
    },
    "band6": {
      "quartz": 0.81,
      "alunite": 0.82,
      "illite": 0.43,
      "chlorite": 0.47,
      "goethite": 0.26,
      "epidote": 0.49,
      "vegetation": 0.19
    },
    "band7": {
      "quartz": 0.82,
      "alunite": 0.72,
      "illite": 0.48,
      "chlorite": 0.63,
      "goethite": 0.22,
      "epidote": 0.67,
      "vegetation": 0.08
    },
  }
}
Need to select all possible two band ratios using Python for every mineral 
("quartz","alunite",      "illite","chlorite","goethite","epidote") 
and “vegetation” value based on a rule: mineral and vegetation band ratios
 must be greater that 1 for vegetatoin reflectances, when for mineral they can 
 be greater or lower than 1 

"""


import json
from itertools import permutations
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import os
from osgeo import gdal
import pandas as pd
from matplotlib import pyplot as plt
import re
from datetime import datetime

#principal component analysis
#https://scikit-learn.org/stable/auto_examples/applications/plot_face_recognition.html#sphx-glr-auto-examples-applications-plot-face-recognition-py
#more on this decomposition parameters
#https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html    

MINERAL_RATIOS = 'mineral_ratios.json'
VEGETATION_RATIOS = 'vegetation_ratios.json'


def pca(X,n_components,m,n): #correct we export components loadings instead of components
  """  Principal Component Analysis
    input: X, matrix with training data stored as flattened arrays in rows
    return: projection matrix (with important dimensions first), variance
    and mean."""

  #from previously run batch image processing
  #pca = PCA(n_components=n_components, svd_solver='randomized',
  #        whiten=True).fit_transform(X)
  pca=PCA(n_components=n_components);
  mean_X = X.mean(axis=0)
  var_X=np.var(X,axis=0)
  #eigenvalues = pca.components_.reshape((n_components,m,n)) #according to the number of components 
  eigenvalues = pca.fit(X).components_.reshape(n_components,m,n);
  # return the PCA matrix [num,:,:], the variance and the mean
  return pca,eigenvalues,var_X,mean_X
    
def mat4pca(imlist): 
    #create matrix to store all flattened images
    immatrix = np.array([np.array(im.flatten())
              for im in imlist],'f') #what is the meaning of the 'f' key?
    return immatrix.T;  #rows for values, cols for variables CORRECTED in 06.02.2021!

def get_pca_image(pca_data,m,n,n_components=1):
    #function to show PCA images
    images={};
    for i in range(0,n_components,1):
        try:
            image=np.array(pca_data[:,i].reshape(m,n));
            images.update({str(i+1):image});
        except:
            print('Can not get {} component. Number of components exceeded'.format(i));
        #image=(np.array(image)-np.min(image))/(np.max(image)-np.min(image));
        
    return images;


#https://scentellegher.github.io/machine-learning/2020/01/27/pca-loadings-sklearn.html
def get_comp_loadings2(pca_model,features,columns):
    loadings = pd.DataFrame(pca_model.components_.T, columns=columns,\
                        index=features)
    eigenvalues=pca_model.explained_variance_; #eigenvalues of the components
    eigenvalues_procents=(eigenvalues/sum(eigenvalues))*100;
        
    row = pd.Series({columns[0]:eigenvalues_procents[0],columns[1]:eigenvalues_procents[1]},name='Eigenvalues')
    loadings = loadings.append(row)
    return loadings;


# The function represents the difference between relationships, exceeding a threshold of 0.5.
def some_action(m_ratio, v_ratio):
    # If it returns True, the pair is kept. 
    # If the mineral ratio is greater than 0.5, we return True.
    return m_ratio > 0.5 


#that procedure picks appropriate band ratios for the minerals and reflectance
def get_band_ratios():
    """
    The function creates caches for calculating combinations of spectral channels in JSON format, in a combinatorial manner.
    """
    # Reflectance data

    data = {
        "Reflectance_spectra": {
            "band1": {"quartz": 0.72, "alunite": 0.69, "illite": 0.26, "chlorite": 0.17, "goethite": 0.07, "epidote": 0.45, "vegetation": 0.04},
            "band2": {"quartz": 0.74, "alunite": 0.73, "illite": 0.31, "chlorite": 0.21, "goethite": 0.07, "epidote": 0.07, "vegetation": 0.05},
            "band3": {"quartz": 0.77, "alunite": 0.81, "illite": 0.38, "chlorite": 0.22, "goethite": 0.17, "epidote": 0.12, "vegetation": 0.09},
            "band4": {"quartz": 0.78, "alunite": 0.82, "illite": 0.34, "chlorite": 0.18, "goethite": 0.18, "epidote": 0.16, "vegetation": 0.06},
            "band5": {"quartz": 0.79, "alunite": 0.83, "illite": 0.31, "chlorite": 0.18, "goethite": 0.19, "epidote": 0.26, "vegetation": 0.55},
            "band6": {"quartz": 0.81, "alunite": 0.82, "illite": 0.43, "chlorite": 0.47, "goethite": 0.26, "epidote": 0.49, "vegetation": 0.19},
            "band7": {"quartz": 0.82, "alunite": 0.72, "illite": 0.48, "chlorite": 0.63, "goethite": 0.22, "epidote": 0.67, "vegetation": 0.08}
        }
    }

    spectra = data["Reflectance_spectra"]
    bands = sorted(spectra.keys(), key=lambda x: int(x.replace('band', '')))
    minerals_list = ["quartz", "alunite", "illite", "chlorite", "goethite", "epidote"]

    veg_collection = []
    min_collection = {m: [] for m in minerals_list}

    # We go through each mineral individually
    for mineral in minerals_list:
        used_mineral_pairs = set() # To ensure that pairs of a mineral do not repeat for the mineral itself
        
        for vb1, vb2 in permutations(bands, 2):
            v_ratio_main = spectra[vb1]["vegetation"] / spectra[vb2]["vegetation"]
            
            # 1. The vegetation filter for its main pair.
            if v_ratio_main > 1:
                remaining_bands = [b for b in bands if b not in (vb1, vb2)]
                
                for mb1, mb2 in permutations(remaining_bands, 2):
                    m_pair_label = f"{mb1}/{mb2}"
                    
                    # We skip it if this mineral has already used this pair of channels.
                    if m_pair_label in used_mineral_pairs:
                        continue
                    
                    # 2. Vegetation filter for mineral pair
                    v_ratio_on_min = spectra[mb1]["vegetation"] / spectra[mb2]["vegetation"]
                    
                    if v_ratio_on_min > 1:
                        m_ratio = spectra[mb1][mineral] / spectra[mb2][mineral]
                        
                        # 3. Fitering
                        if some_action(m_ratio, v_ratio_on_min):
                            v_pair_label = f"{vb1}/{vb2}"
                            
                            # Save into mineral band ratios collection
                            min_entry = {
                                "mineral_pair": m_pair_label,
                                "mineral_ratio": round(m_ratio, 3),
                                "v_ratio_on_this_pair": round(v_ratio_on_min, 3),
                                "linked_veg_pair": v_pair_label
                            }
                            min_collection[mineral].append(min_entry)
                            used_mineral_pairs.add(m_pair_label)
                            
                            # Save into vegetation band ratios collection
                            veg_collection.append({
                                "veg_pair": v_pair_label,
                                "veg_ratio": round(v_ratio_main, 3),
                                "used_for_mineral": mineral
                            })

    # Check-up
    minerals = ["quartz","alunite","illite","chlorite","goethite","epidote"]
    used_minerals = []
    for mineral in minerals:
        print(f"For {mineral} there are potential band ratios: {len(min_collection[mineral])}")
        for i in range(len(min_collection[mineral])):
            if min_collection[mineral][i]['mineral_pair'] not in used_minerals:
                print(f"Result (index {i}):", min_collection[mineral][i])
                used_minerals.append(min_collection[mineral][i]['mineral_pair'])

    for veggie in veg_collection:
        print(f"Vegetation {veggie['veg_pair']}: {veggie['veg_ratio']}")
    
    print('Saving mineral ratios...')
    try:
        with open(MINERAL_RATIOS, "w", encoding="utf-8") as file:
            json.dump(min_collection, file, indent=4, ensure_ascii=False)
        with open(VEGETATION_RATIOS, "w", encoding="utf-8") as file:
            json.dump(veg_collection, file, indent=4, ensure_ascii=False)
    except:
        print('Can not save data, reason unknown')
    
    return min_collection

def generate_landsat_basename(path, row, date_str, format_type='new'):
    """
    Clear Landsat base name
    """
    # Clear 'band' substring, if there
    dt = datetime.strptime(date_str, '%Y-%m-%d')

    if format_type == 'new':
        # Format: LC08_L1TP_105029_20160522_20170221_01_T1_B6
        date_iso = dt.strftime('%Y%m%d')
        # The new format usually contains additional processing dates,
        # here we use the shooting date as a blank for simplicity
        return f"LC08_L1TP_{path}{row}_{date_iso}_{date_iso}_02_T1"

    else:
        # Format: LC81050292016143LGN00_B6
        # %j - Julian day
        date_julian = dt.strftime('%Y%j')
        return f"LC8{path}{row}{date_julian}LGN00"
    

def parse_landsat_full(filename, default_val="000", default_date="1970-01-01"):
    """
    Function for using data from a Landsat filename
    """
    # Old format pattern (LC81050292016143..._B6)
    # Naming: 1:Path, 2:Row, 3:Year+JulianDay, 4:Band
    pattern_old = r'L[CO]8(\d{3})(\d{3})(\d{7}).*?_B(\d+)'
    
    # New format pattern (LC08_L1TP_105029_20160522..._B1)
    pattern_new = r'L[CO]\d{2}_.{4}_(\d{3})(\d{3})_(\d{8}).*?_B(\d+)'

    # 1. Check old format
    match = re.search(pattern_old, filename)
    if match:
        path, row, date_julian, band = match.groups()
        try:
            date_str = datetime.strptime(date_julian, '%Y%j').strftime('%Y-%m-%d')
        except ValueError:
            date_str = default_date
        return path, row, date_str, f"band{band}"

    # 2. Check new format
    match = re.search(pattern_new, filename)
    if match:
        path, row, date_iso, band = match.groups()
        try:
            date_str = datetime.strptime(date_iso, '%Y%m%d').strftime('%Y-%m-%d')
        except ValueError:
            date_str = default_date
        return path, row, date_str, f"band{band}"

    # 3. Use dafault values if failed to parse
    return default_val, default_val, default_date, "band0"


def saveGeoTiff(raster,filename,gdal_object,ColMinInd,RowMinInd,nodata_value=0): #ColMinInd,RowMinInd - start row/col for cropped images
    meas=np.shape(raster)
    rows=meas[0]; cols=meas[1]; 
    if(len(meas)==3):
        zs=meas[2];
    else:
        zs=1;
    print("Saving "+filename)    
    driver = gdal.GetDriverByName("GTiff")

    # Compression parameters
    # LZW effectively compress Float64 
    # PREDICTOR=3 compression optimized for floats
    options = [
        'COMPRESS=LZW',
        'PREDICTOR=3',
        'TILED=YES'
    ]

    outdata = driver.Create(filename, cols, rows, zs, gdal.GDT_Float64,options=options)
    (start_x,resx,zerox,start_y,zeroy,resy)=gdal_object.GetGeoTransform()
    outdata.SetGeoTransform((start_x+(resx*ColMinInd),resx,zerox,start_y+(resy*RowMinInd),zeroy,resy));
    outdata.SetProjection(gdal_object.GetProjection()) #sets same projection as input
    #write bands
    if zs>1:
        for b in range(0,zs):
            outdata.GetRasterBand(b+1).WriteArray(raster[:,:,b])
            outdata.GetRasterBand(b+1).SetNoDataValue(nodata_value) ##if you want these values transparent
    else:
        outdata.GetRasterBand(1).WriteArray(raster) #write single value raster
        outdata.GetRasterBand(1).SetNoDataValue(nodata_value)
    outdata.FlushCache() ##saves


def kmeans_sort(img,do_reverse=0,n_clusters=2):
    #applies KMeans classification
    #data=np.array([0,1,2,3,45,46,47,100,101,102]).reshape(-1,1);
    shape=np.shape(img);
    data=img.flatten().reshape(-1,1);
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(data)
    
    classes = kmeans.labels_
    
    mean_values=np.array([]);
    class_arr=np.arange(0,max(classes)+1);
    #arrange labels according to their values
    for label in class_arr:
        mean_values=np.append(mean_values,np.mean(data[classes==label]));        
    
    #sort values of array classes according to their labels
    if do_reverse==1:
        inds = mean_values[::-1].argsort();    
    else:
        inds = mean_values.argsort();
    new_classes=np.zeros(classes.shape,dtype=np.int8);
    for ind,new_val in zip(inds,range(0,len(inds))):
        new_classes[classes==ind]=new_val;
    
    new_classes=np.reshape(new_classes,shape);
    return new_classes; 


def dpca(dir_products_path:str,dir_cropped_path:str,fileext='tif',dempostfix='dem',min_bands_json='mineral_ratios.json'):
    """
    dir_products_path - path for output files
    dir_cropped_path - path for input files 
    """
    #files for processing, input and output directory

    cov_ratios_dpca_save_iron=os.path.join(dir_products_path,"ratios_dpca_cov_stat_iron.xls");
    cov_ratios_dpca_save_clay=os.path.join(dir_products_path,"ratios_dpca_cov_stat_clay.xls");

    loadings_dpca_save_iron=os.path.join(dir_products_path,"loadings_stat_iron.xls");
    loadings_dpca_save_clay=os.path.join(dir_products_path,"ratios_dpca_cov_stat_clay.xls");
    loadings_filemask='loadings_DPCA_{}.xls';
    loadings_filemask2='loadings_DPCA_2_{}.xls';
    variance_filemask='variance_DPCA_{}.xls';

    band_number_inname='_b%N%.' #%N% - for band number e.g. LC81050292016143LGN00_B6.TIF NOT A CASE SENSITIVE
    band_number_inname=band_number_inname.lower();
    ColMinInd=0; RowMinInd=0; #because we work on already cropped pictures

    #list of bands to be used in computations
    bands=[2,3,4,5,6,7];

    DO_KMEANS=False; #if this flag set to FALSE image will be neglected (*-1), if TRUE
                    #k-means classififed

    DO_TOPO_MASK=True; #do topographic masking (for water areas)

    DO_NORM_NEG_NORM=True; #normalize data 0 - 1, scale data 1 - 0  
                            #in case of negative loadings 

    NDVI_CLASS_MASK=0; #ndvi class to be excluded from computations -1 if unused


    #2 data processing
    bands={}
    try:
        for file in os.listdir(dir_cropped_path):         #exclude 3band tif files
            #select topo
            if file.lower().endswith("."+fileext.lower()) and dempostfix.lower() in file.lower():
                gdal_object = gdal.Open(os.path.join(dir_cropped_path,file)) #as new gdal_object was created, no more ColMinInd,RowMinInd
                bands[dempostfix]=gdal_object.GetRasterBand(1).ReadAsArray();
                TOPO_MASK_NODATA = gdal_object.GetRasterBand(1).GetNoDataValue()

            #select landsat band
            elif file.lower().endswith("."+fileext.lower()) and dempostfix.lower() not in file.lower(): 
                path, row, date_str,bandNum = parse_landsat_full(file, default_val="000", default_date="1970-01-01")

                if path!='000' and row!='000':
                    gdal_object = gdal.Open(os.path.join(dir_cropped_path,file)) #as new gdal_object was created, no more ColMinInd,RowMinInd
                    bands[bandNum]=gdal_object.GetRasterBand(1).ReadAsArray() ;
                    print(file+" was processed");
    except(FileNotFoundError):
            print("Input image folder doesn\'t exist or file was not found...");
            return 

    
    #Compute Em.J.M. Carranza band rations and their PCs
    #mineral band ratios were updated according to Prof. Carranza 

    m,n=bands[bands.keys()[0]].shape;

    #loading possible combinations from the cache (file mineral_ratios.json)
    data = load_json_bands_data(min_bands_json)
    minerals_list = get_minerals_list_json(data)

    if not DO_TOPO_MASK:
        #ndvi ratio for area classes
        ndvi=(bands['band5']-bands['band4'])/(bands['band5']+bands['band4']);

        #nvdi classes using K-means
        ndvi_classes=kmeans_sort(ndvi,do_reverse=0,n_clusters=4);
                
        #show k-means classes ndvi
        colors = [(0, 0, 1), (0, 0.5, 0.2), (0, 1, 0),(1, 0, 0)]  # R -> G -> B
        n_bins = 4  # Discretizes the interpolation into binsformat('')+pathrowfolder+"_"+datefolder+"
        cmap_name = 'my_list'
        cm = LinearSegmentedColormap.from_list(cmap_name, colors, N=n_bins);

        #tell the colorbar to tick at integers
        im=plt.imshow(ndvi_classes,cmap=cm);
        plt.colorbar(im,ticks=np.arange(np.min(ndvi_classes),np.max(ndvi_classes)+1))

        plt.savefig(os.path.join(dir_products_path,'ndvi_classes.png'),dpi=300);
        plt.savefig(os.path.join(dir_products_path,'ndvi_classes.svg'),dpi=300);
        plt.close();

        #save ndvi classes
        saveGeoTiff(ndvi_classes,os.path.join(dir_products_path,f'ndvi_classes{path}{row}_{date_str}.tif'),gdal_object,ColMinInd,RowMinInd);

    
    
    #assessment of channel standart deviation, selecting channels with the higher deviation
    minerals={}
    vegetation={}
    used_mineral_ratios = [] #list to store used mineral ratios, that were kept 



    for mineral in minerals_list:
        for i in range(len(data[mineral])):
            mineral_band_ratio = data[mineral][i]["mineral_pair"]
            separated_bands_minerals= mineral_band_ratio.split('/')
            linked_veg_pair=data[mineral][i]["linked_veg_pair"]
            separated_bands_vegetation=linked_veg_pair.split('/')

            minerals.update({mineral:bands[separated_bands_minerals[0]]/bands[separated_bands_minerals[1]]});
            vegetation.update({mineral:bands[separated_bands_vegetation[0]]/bands[separated_bands_vegetation[1]]});

    #compute leaves/minerals DPC's
    #compute DPCA for minerals vs vegetation 5/4
    #1 flatten image matrixes
    print("Started to compute DPCA for the iron oxides and clay minerals...")
    print("Flatten image matrix...")

    mineral_vegetation_flat={};  #compute flattened matrix for all minerals
    for key in minerals_list:
        mineral_vegetation_flat.update({key:mat4pca((minerals[key],vegetation[key]))});     

    #2 compute PCA for minerals
    pca_models_minerals={};
    pca_dpca_minerals={};


    #selecting/masking non-water NDVI pixels 
    for key in minerals_list:
        pca_models_minerals.update({key:PCA(n_components=2)});
        #updates of 2021/02/26
        if not DO_TOPO_MASK:
            mask_index=(ndvi_classes.flatten()!=NDVI_CLASS_MASK); #mask "no-water"
        
        if DO_TOPO_MASK:
            mask_index=(bands[dempostfix].flatten()!=TOPO_MASK_NODATA);

        blank_table_pca=np.zeros(mineral_vegetation_flat[key].shape);
        table4pca=mineral_vegetation_flat[key][mask_index]
        found_pca=pca_models_minerals[key].fit_transform(table4pca); #table with pca inside    
        blank_table_pca[mask_index]=found_pca; #place computed DPCA into blank table, mask stays zero
        pca_dpca_minerals.update({key:blank_table_pca});
        

    #3 loadings for minerals/vegetation (eigenvectors/eigenvalues)
    
    loadings_minerals={};
    for key in [*minerals]:
        #split 30% of data set for computation loadings
        #mineral_veg_train, mineral_veg_test = train_test_split(mineral_vegetation_flat[key],\
        #                                test_size=0.66, random_state=42);
        loadings = get_comp_loadings2(pca_models_minerals[key],\
            features=[key,'vegetation'],columns=['DPC1', 'DPC2']);
        print(loadings);                              
        loadings_minerals.update({key:loadings}); #    

    #4 DPCA for minerals
    DPCA_minerals={};
    for key in [*minerals]:
        pca_image=get_pca_image(pca_dpca_minerals[key],m,n,n_components=2);
        pca_image_classes={};
        for pckey in pca_image: #classify PCA images
            if loadings_minerals[key][('DPC'+pckey)][key]<0:
                do_reverse=1; #if loading is negative multiply -1
                multiplier=-1;
            else:
                do_reverse=0;
                multiplier=1; #NO negation
            
            
            if DO_KMEANS==False:
                #no K-means classification
                if DO_NORM_NEG_NORM==False:
                    img_class=pca_image[pckey]*multiplier;
                else:
                    if multiplier==1:
                        img_class=(pca_image[pckey]-np.min(pca_image[pckey]))/\
                            (np.max(pca_image[pckey])-np.min(pca_image[pckey]));
                    else: #multiplier==-1
                        img_class=(np.max(pca_image[pckey])-pca_image[pckey])/\
                            (np.max(pca_image[pckey])-np.min(pca_image[pckey]));
            else:
                #uncomment line below if you wish k-means classification
                img_class=kmeans_sort(pca_image[pckey],do_reverse=do_reverse,n_clusters=10);
            
            #img_class=mean_shift_sort(pca_image[pckey],do_reverse=do_reverse);
            pca_image_classes.update({pckey:img_class});
            #pca_image_classes.update({pckey:get_image_classes(pca_image[pckey])});
        DPCA_minerals.update({key:pca_image_classes});    


    #5 save loadings
    for key in [*minerals]:
        loadings_minerals[key].to_excel(os.path.join(dir_products_path,\
                        loadings_filemask.format(key)),index=True);
        
                        
    #6 save DPCs into geotiff and matplotlib image
    for key in [*minerals]:
        for n in DPCA_minerals[key]: #save normalize values of DPCA matrix
            #apply negation 
            img=DPCA_minerals[key][n];
            
            
            saveGeoTiff(img,os.path.join(dir_products_path,"DPC{}_{}_".format(n,key)+pathrow+"_"+date+".tif"),gdal_object,ColMinInd,RowMinInd);        

#function for loading band ratios
def load_and_parse_json(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return data
    except FileNotFoundError:
        print(f"Error: File {file_path} not found.")
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e.msg} (line {e.lineno}, column {e.colno})")
    except Exception as e:
        print(f"Unknown error: {e}")
    return None

def load_json_bands_data(fname):
    """
    This function should generate a cache of channel relationships if the cache is missing or cannot be parsed.
    """
    data = load_and_parse_json(fname)

    if not data:
        print('File is not available')
        print('Compute mineral band ratios...')
        print('Saving cached file...')
        data = get_band_ratios()
    return data

def get_minerals_list_json(data):
    """
    Gets minerals list from the loaded json structure
    """
    return [*data.keys()]


def get_mineral_veg_pairs(data,mineral_name):
    """
    extracts mineral and vegetation band pairs, return possible list
    """
    print('Unique minerals in a band ratios data structure:',*data.keys())
    print('test output for fist 5 mineral and vegetsation ratios: for 1 mineral (quartz)')
    
    if mineral_name not in [*data.keys()]:
        print('Mineral name is not found')
        return 

    return data[mineral_name][:]["mineral_pair"],data[mineral_name][:]["linked_veg_pair"]

def get_bands_splitted(bandratio):
    """
    split band ratios like band1/band2 with / sign
    returns tuple ("band1","band2")
    """
    return tuple(bandratio.split('/'))


#UNTESTED FUNCTION FOR PROCESSING
def process_pca_with_negation(A, B, C=None, A_name='vegetation', B_name='mineral'):
    """
    A: vegetation channel ratio matrix
    B: mineral channel ratio matrix
    C: valid land pixel mask
    """
    if C is None:
        C = np.ones(A.shape, dtype=bool)

    # Creating flat columns under a mask
    a_masked = A[C].reshape(-1, 1)
    b_masked = B[C].reshape(-1, 1)
    
    # Combine into a two-feature matrix (n_samples, 2)
    combined_data = np.hstack([a_masked, b_masked])

    # Train PCA
    pca = PCA(n_components=2)
    coords = pca.fit_transform(combined_data) 
    
    eigenvalues = pca.explained_variance_ 
    loadings = pca.components_  # Loadings matrix (2, 2)

    # Results preparation for CSV logging
    data_table = {
        'Eigenvalue': np.round(eigenvalues, 5),
        'Variance_pct': np.round(pca.explained_variance_ratio_ * 100, 2),
        f'Loading_{A_name}': loadings[:, 0], # Index 0 - vegetation
        f'Loading_{B_name}': loadings[:, 1], # INdex 1 - mineral
    }
    df_output = pd.DataFrame(data_table, index=['PC1', 'PC2'])

    # We need a component where the signs of the loadings are OPPOSITE
    # That is, their product is less than zero: Loading_veg * Loading_min < 0
    target_idx = None
    for idx in (0, 1):
        if loadings[idx, 0] * loadings[idx, 1] < 0:
            target_idx = idx
            break
            
    # If, by any chance, the signs are the same in both cases (a data anomaly), we revert to the traditional approach: using PC2 (minimum variance).
    if target_idx is None:
        target_idx = np.argmin(eigenvalues)

    # We extract the pure vector of projections for the selected component.
    target_component_data = coords[:, target_idx]

    # We need the contribution of the MINERAL to the final brightness to be POSITIVE. 
    # Look at the sign of the mineral's loading (Loading_mineral) in the selected component.
    multiplier = 1.0
    if loadings[target_idx, 1] < 0:
        multiplier = -1.0
        
    # We apply sign alignment directly to the component.
    final_mineral_signal = target_component_data * multiplier

    # Map the result back onto the raster grid using a mask
    result = np.zeros_like(B, dtype=float)
    result[C] = final_mineral_signal

    return result, df_output


def dpca_new(dir_products_path:str,dir_cropped_path:str,fileext='tif',dempostfix='topo',min_bands_json='mineral_ratios.json'):
    """
    dir_products_path - path for output files
    dir_cropped_path - path for input files 

    process scene dir at a time

    """

    band_number_inname='_b%N%.' #%N% - for band number e.g. LC81050292016143LGN00_B6.TIF NOT A CASE SENSITIVE
    band_number_inname=band_number_inname.lower();
    ColMinInd=0; RowMinInd=0; #because we work on already cropped pictures

    #list of bands to be used in computations
    DO_TOPO_MASK=True; #do topographic masking (for water areas)
    NORMALIZE_OUTPUT = True; #normalize mineral ratio output

    if not os.path.exists(dir_products_path):
        os.makedirs(dir_products_path)


    print(os.listdir(dir_cropped_path))
    #2 data processing
    bands={}
    LANDSAT_MASK_NODATA = None
    try:
        for file in os.listdir(dir_cropped_path):         #exclude 3band tif files
            #select topo
            if file.lower().endswith("."+fileext.lower()) and dempostfix.lower() in file.lower():
                gdal_object = gdal.Open(os.path.join(dir_cropped_path,file)) #as new gdal_object was created, no more ColMinInd,RowMinInd
                bands[dempostfix]=gdal_object.GetRasterBand(1).ReadAsArray();
                TOPO_NODATA = gdal_object.GetRasterBand(1).GetNoDataValue()
                

            #select landsat band
            elif file.lower().endswith("."+fileext.lower()) and dempostfix.lower() not in file.lower(): 
                path, row, date_str,bandNum = parse_landsat_full(file, default_val="000", default_date="1970-01-01")
                

                if path!='000' and row!='000':
                    gdal_object = gdal.Open(os.path.join(dir_cropped_path,file)) #as new gdal_object was created, no more ColMinInd,RowMinInd
                    band_data = gdal_object.GetRasterBand(1).ReadAsArray()
                    bands[bandNum] = band_data

                    if bandNum == 'band1':
                    
                        # We obtain the official NoData value from the metadata.
                        nodata_val = gdal_object.GetRasterBand(1).GetNoDataValue()

                        # Create a mask for valid data
                        # For Landsat, black margins are either nodata_val or simply 0
                        if nodata_val is not None:
                            mask = (band_data != nodata_val) & (band_data > 0)
                        else:
                            mask = (band_data > 0)

                        # No data and a mask are saved:
                        
                        LANDSAT_MASK_NODATA = mask

                    print(file+" was processed");
                    try:
                        base_band_name = file.split('_')[0]
                    except:
                        base_band_name = generate_landsat_basename(path, row, date_str, format_type='old')
    except(FileNotFoundError):
            print("Input image folder doesn\'t exist or file was not found...");
            return 

    print(bands.keys())
    #Compute J.Carranza band rations and their PCs
    #mineral band ratios were updated according to Prof. Carranza letter of 2021/02/05
    m,n=bands[[*bands][0]].shape;

    #getting possible band ratios from the cache file (file mineral_ratios.json)
    data = load_json_bands_data(min_bands_json)
    minerals_list = get_minerals_list_json(data)
    used_mineral_ratios = [] #band ratios of minerals successfully used for DPCA

    topo_mask = bands[dempostfix]!=TOPO_NODATA #exclude nodata values
    if LANDSAT_MASK_NODATA is not None:
        topo_landsat_mask = topo_mask & LANDSAT_MASK_NODATA
    else:
        topo_landsat_mask = topo_mask
    for mineral in minerals_list:
        print(f'get {mineral} and vegetation pairs...')
        for i in range(len(data[mineral])):
            print(f'Computations for {mineral}')
            mineral_pair = data[mineral][i]["mineral_pair"]
            if mineral_pair in used_mineral_ratios:
                continue #skip to the next mineral ratio
            min_band1,min_band2 = mineral_pair.split('/')
            veg_band1,veg_band2 = data[mineral][i]["linked_veg_pair"].split('/')
            

            mineral_ratio = np.divide(bands[min_band1], bands[min_band2], out=np.zeros_like(bands[min_band1], dtype=float), where=bands[min_band2]!=0)
            vegetation_ratio = np.divide(bands[veg_band1], bands[veg_band2], out=np.zeros_like(bands[veg_band1], dtype=float), where=bands[veg_band2]!=0)

            #do DPCA mineral_dpc - fileter mineral componentmask
            mineral_dpc,df_output = process_pca_with_negation(vegetation_ratio, mineral_ratio, topo_landsat_mask)

            

            if mineral_dpc is not None and df_output is not None:    
                df_output.to_csv(os.path.join(dir_products_path,f'{base_band_name}_{mineral}_loading.csv'))

                print(f'Saving {mineral} matrix to {dir_products_path}')
                geotiffn = os.path.join(dir_products_path,f'{base_band_name}_{mineral}.tif')
                mineral_dpc[~topo_landsat_mask] = 0

                if NORMALIZE_OUTPUT:
                    if mineral_dpc is not None:
                        print('Normalize output data...')
                        valid_data = mineral_dpc[topo_landsat_mask]
                        
                        v_min = valid_data.min()
                        v_max = valid_data.max()
                        
                        # To avoid zero division error if all pixels are same
                        if v_max - v_min > 0:
                            mineral_dpc[topo_landsat_mask] = (valid_data - v_min) / (v_max - v_min)
                            
                used_mineral_ratios.append(mineral_pair)
                saveGeoTiff(mineral_dpc,geotiffn,gdal_object,ColMinInd=0,RowMinInd=0,nodata_value=0)
                break #proceed to the next mineral species
            else:
                continue

                 

## Test launch of the DPC method

In [ ]:
landsat_folder = os.path.join('..','files','tif','Landsat_Test_Input')
landsat_product_folder = os.path.join('..','files','tif','Landsat_Test_Input','products')
dpca_new(landsat_product_folder,landsat_folder,fileext='tif',dempostfix='topo',min_bands_json='mineral_ratios.json')


## Model adjustment and training

In [ ]:
def get_training_dataset(processing_dir: str, active_model_path: str, aoi_path: str):
    """
    Performs a spatial SQL slice from an SQLite database
    based on the metric frames (training) stored in the aoi.json file.
    """
    with open(active_model_path, 'r', encoding='utf-8') as f:
        am = json.load(f)
    with open(aoi_path, 'r', encoding='utf-8') as f:
        aoi = json.load(f)
        
    t_zone = aoi["training"]
    db_path = Path(processing_dir) / "data.db"
    
    # Read project metadata from modproj to convert meters into grid indices.
    conn = sqlite3.connect(str(db_path))
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM modproj LIMIT 1")
    proj_meta = dict(cursor.fetchone())
    
    ref_xmin = proj_meta['ref_xmin']
    ref_ymax = proj_meta['ref_ymax']
    ref_xres = proj_meta['ref_xres']
    ref_yres = proj_meta['ref_yres']
    
    # Convert the angular dimensions of the selected AOI into indices for the `modlayers` matrix (as pure floats).
    print('Convert the angular dimensions of the selected AOI into indices for the `modlayers` matrix (as pure floats).')
    start_col = round((t_zone["xmin_meters"] - ref_xmin) / ref_xres)
    end_col = round((t_zone["xmax_meters"] - ref_xmin) / ref_xres)
    start_row = round((ref_ymax - t_zone["ymax_meters"]) / ref_yres)
    end_row = start_row + round((t_zone["ymax_meters"] - t_zone["ymin_meters"]) / ref_yres)

    print('Compiling a list of columns for an SQL query')
    target_col = am["target_variable"]
    predictors_list = list(am["predictors"].keys())
    all_cols = ["rowind", "colind"] + [target_col] + predictors_list
    cols_str = ", ".join([f'"{c}"' for c in all_cols])

    query = f"""
        SELECT {cols_str} FROM modlayers 
        WHERE rowind >= {min(start_row, end_row)} AND rowind <= {max(start_row, end_row)}
          AND colind >= {min(start_col, end_col)} AND colind <= {max(start_col, end_col)}
    """
    df = pd.read_sql(query, conn)
    conn.close()
    
    # Replace system NoData values (-1.0) with a user-defined value (default is 0.0)
    nodata_val = am.get("nodata_replacement_value", 0.0)
    for p in predictors_list:
        df[p] = df[p].replace(-1.0, nodata_val)
        
    return df, am, proj_meta



def execute_model_training(processing_dir: str, config_data: dict):
    """
    Executes a full training cycle for MaxEnt (OneClassSVM) or a Perceptron,
    with a 70/30 data split and calculation of ROC-AUC and PDP on the 30% validation set.
    """
    start_time = time.clock() if hasattr(time, 'clock') else time.time()
    
    active_json_path = Path(".") / "active_model.json"
    aoi_path = Path(processing_dir) / "aoi.json"
    
    # Spatial data extraction
    df, am, proj_meta = get_training_dataset(processing_dir, str(active_json_path), str(aoi_path))

    print(f'Totally {len(df)} records fetched')
    
    X_cols = list(am["predictors"].keys())
    y_col = am["target_variable"]
    model_type = am["selected_model"]
    
    # To calculate the ROC-AUC on the validation set, we need binary ground-truth labels in any case.
    df['TRUE_LABELS'] = (df[y_col] > 0).astype(int)
    
    if model_type == "MaxEnt_SVM":
        # Filtering the dataset: for One-Class, we include ONLY those pixels where occurrence points are present!
        df_presence = df[df[y_col] > 0]
        if len(df_presence) < 5:
            return {"status": "error", "message": f"Occurence points are too few ({len(df_presence)}) for MaxEnt!"}
            
        X_data = df_presence[X_cols]
        # We split the ore-related data points: 70% for training, 30% for anomaly validation.
        X_train, X_val = train_test_split(X_data, test_size=0.3, random_state=42)
        
        print('Adjust OneClassSVM for Presence-Only analysis')
        base_clf = OneClassSVM(kernel='rbf')
        param_grid = {
            'classifier__nu': [0.001, 0.01, 0.03, 0.05, 0.1, 0.15, 0.2],
            'classifier__gamma': ['scale', 'auto', 0.001, 0.01, 0.05, 0.1, 0.5, 1, 5, 10]
        }
    else:
        # A Perceptron (MLP) requires a classic binary dataset (0 и 1)
        X_data = df[X_cols]
        y_data = df['TRUE_LABELS']
        X_train, X_val, y_train, y_val = train_test_split(X_data, y_data, test_size=0.3, random_state=42)
        
        base_clf = MLPClassifier(max_iter=500, random_state=42)
        param_grid = {
            'classifier__hidden_layer_sizes': [(50,), (100,)],
            'classifier__activation': ['tanh', 'relu']
        }

    # Taking manual parameters from the web form, if provided.
    user_params = config_data.get("params", {})
    if user_params:
        for k, v in user_params.items():
            param_grid[f'classifier__{k}'] = [v] if not isinstance(v, list) else v

    # Assemble a Pipeline
    preprocessor = build_preprocessor(am)
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', base_clf)])
    
    # Use GridSearchCV or direct training
    print('Use GridSearchCV or direct training')
    if config_data.get("use_gridsearch", False):
        test_sz = 0.01 if len(X_train) > 500 else 0.1
        
        if model_type == "MaxEnt_SVM":
            # 1. We take a subsample of the training ore points (class 1)
            X_opt_presence = train_test_split(X_train, test_size=test_sz, random_state=42)[0]
            if len(X_opt_presence) == 0:
                X_opt_presence = X_train
            y_opt_presence = np.ones(len(X_opt_presence), dtype=int)
            
            # 2. We cut out the background using the TRUE LABELS mask, which contains all the pixels representing empty rock.
            global_df = df.copy() 
            df_background = global_df[global_df['TRUE_LABELS'] == 0]
            
            # Tracing of a progress
            print(f"[GIS GRID TRACING] Findings for GridSearch (class 1):: {len(X_opt_presence)}")
            print(f"[GIS GRID TRACING] Total ACTUAL barren rock background (TRUE_LABELS==0): {len(df_background)}")
            
            # From the actual volume of the query, we extract a number of random pixels exactly equal to the number of findings we have
            X_opt_background = df_background[X_cols].sample(n=len(X_opt_presence), random_state=42)
            y_opt_background = np.zeros(len(X_opt_background), dtype=int)
            
            # Standardizing the sizes of presence indicators
            y_opt_presence = np.ones(len(X_opt_presence), dtype=int)
            
            # 3. Combine into a binary matrix for grid validation
            X_grid_cv = pd.concat([X_opt_presence, X_opt_background])
            y_grid_cv = np.concatenate([y_opt_presence, y_opt_background])
            
            
            #  Let’s shuffle the indices so that the zeros and ones are distributed evenly!
            from sklearn.utils import shuffle
            X_grid_cv, y_grid_cv = shuffle(X_grid_cv, y_grid_cv, random_state=42)
            
            print(f"Final matrix X_grid_cv: {X_grid_cv.shape}")
            
            # We use StratifiedKFold to ensure that both classes are present in each fold.
            from sklearn.model_selection import StratifiedKFold
            cv_strat = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
            
            grid = GridSearchCV(
                pipeline, param_grid, cv=cv_strat, n_jobs=-1,  
                scoring='roc_auc', verbose=3, refit=True, error_score='raise'
            )
            grid.fit(X_grid_cv, y_grid_cv)
            
        else:
            # Data preparation for the MLPClassifier
            # Unpack all 4 arrays correctly, and take X_opt from the training portion [0]
            X_opt, _, y_opt, _ = train_test_split(
                X_train, y_train, test_size=test_sz, random_state=42, stratify=y_train
            )
            
            grid = GridSearchCV(
                pipeline, param_grid, cv=3, n_jobs=-1, 
                scoring='roc_auc', verbose=3, refit=True
            )
            grid.fit(X_opt, y_opt)
            
        best_pipeline = grid.best_estimator_
        final_params = grid.best_params_
    else:
        single_params = {}
        for k, v in param_grid.items():
            single_params[k] = v[0] if isinstance(v, list) else v
        pipeline.set_params(**single_params)
        best_pipeline = pipeline
        final_params = single_params

    # We fine-tune the final pipeline on the entire training dataset.
    print('Fine-tune the final pipeline on the entire training dataset.')
    best_pipeline.fit(X_train, *( [y_train] if model_type != "MaxEnt_SVM" else [] ))
    
    # Create an isolated directory for the model, using its unique UUID.
    print('Create an isolated directory for the model, using its unique UUID')
    model_uuid = am["model_id"]
    output_model_dir = Path(processing_dir) / "trained_models" / model_uuid
    output_model_dir.mkdir(parents=True, exist_ok=True)

    # 4. CALCULATION OF VALIDATION METRICS ON 30% OF THE DATA (Refactored for evaluation purposes)
    # To calculate a proper ROC-AUC on the validation set, we need a sample that contains both zeros and ones. 
    # Therefore, we create a subset from the original dataframe using the indices that were not included in X_train.
    print('CALCULATION OF VALIDATION METRICS ON 30% OF THE DATA (Refactored for evaluation functionality)')
    df_val_full = df.loc[~df.index.isin(X_train.index)].copy()
    
    X_val_full = df_val_full[X_cols]
    y_val_true = df_val_full['TRUE_LABELS'].to_numpy()
    
    if model_type == "MaxEnt_SVM":
        val_raw_scores = best_pipeline.score_samples(X_val_full)
        min_vs, max_vs = val_raw_scores.min(), val_raw_scores.max()
        df_val_full['PROSPECTIVITY'] = (val_raw_scores - min_vs) / (max_vs - min_vs + 1e-8)
    else:
        df_val_full['PROSPECTIVITY'] = best_pipeline.predict_proba(X_val_full)[:, 1]
        
    val_scores_arr = df_val_full['PROSPECTIVITY'].to_numpy()
    
    # Computation of  ROC-AUC
    print('Computation of ROC-AUC')
    fpr, tpr, _ = roc_curve(y_val_true, val_scores_arr)
    roc_auc_val = sklearn_auc(fpr, tpr)
    
    # Rendering and saving the validation ROC curve
    print('Rendering and saving the validation ROC curve')
    fig_roc, ax_roc = plt.subplots(figsize=(6, 5))
    ax_roc.plot(fpr, tpr, color='darkorange', linewidth=2, label=f'ROC curve (AUC = {roc_auc_val:.4f})')
    ax_roc.plot([0, 1], [0, 1], color='navy', linewidth=2, linestyle='--')
    ax_roc.set_xlim([0.0, 1.0])
    ax_roc.set_ylim([0.0, 1.05])
    ax_roc.set_xlabel('False Positive Rate (FPR)')
    ax_roc.set_ylabel('True Positive Rate (TPR)')
    ax_roc.set_title('Validation ROC curve (30% dataset)')
    ax_roc.legend(loc="lower right")
    ax_roc.grid(True, linestyle=':', alpha=0.6)
    plt.savefig(output_model_dir / "roc_curve_validation.png", dpi=150, bbox_inches='tight')
    plt.close(fig_roc)

    # Generating PDP response curves on validation data
    print('Generating PDP response curves on validation data')
    generate_pdp_curves(best_pipeline, df_val_full, X_cols, output_model_dir)

    # Generate a continuous probability forecast for the ENTIRE clipped area of interest (for the map).
    print('Generate a continuous probability forecast for the ENTIRE clipped area of interest (for the map)')
    X_full = df[X_cols]
    if model_type == "MaxEnt_SVM":
        raw_scores = best_pipeline.score_samples(X_full)
        min_s, max_s = raw_scores.min(), raw_scores.max()
        full_preds = (raw_scores - min_s) / (max_s - min_s + 1e-8)
    else:
        full_preds = best_pipeline.predict_proba(X_full)[:, 1]
    
    # Dump the entire trained pipeline
    print('Dump the entire trained pipeline')
    pkl_path = output_model_dir / am["model_filename"]
    with open(pkl_path, 'wb') as f:
        pickle.dump(best_pipeline, f)

    # ALGHOFAILI METHOD BLOCK ON THE VALIDATION SET (30%)
    print("[GIS Tracing] Launching the Algofaili method for assessing predictor importance...")
    top_predictors_str = "Not enough data"
    pixel_classes_val = np.zeros(len(df_val_full), dtype=int)
    
    if model_type == "MaxEnt_SVM":
        train_scores = best_pipeline.score_samples(X_train)
        threshold = np.percentile(train_scores, 5) 
        min_vs, max_vs = val_raw_scores.min(), val_raw_scores.max()
        norm_threshold = (threshold - min_vs) / (max_vs - min_vs + 1e-8)
        pixel_classes_val[df_val_full['PROSPECTIVITY'] >= norm_threshold] = 1
    else:
        pixel_classes_val[df_val_full['PROSPECTIVITY'] >= 0.5] = 1

    dict_out = {'num': [], 'predictors': [], 'weights': [], 'class': []}
    num = 0
    X_val_pdp = df_val_full[X_cols]
    
    for cl in [0,1]:
        clf_rf = RandomForestClassifier(n_estimators=50, max_depth=8, n_jobs=1, random_state=1)
        series = np.zeros(len(X_val_pdp), dtype=int)
        series[pixel_classes_val == cl] = 1
        if np.sum(series) > 1:
            clf_rf.fit(X_val_pdp.values, series)
            for ind in np.argsort(clf_rf.feature_importances_)[::-1]:
                dict_out['num'].append(num)
                dict_out['predictors'].append(X_cols[ind])
                dict_out['weights'].append(round(float(clf_rf.feature_importances_[ind]), 4))
                dict_out['class'].append("PERSPECTIVE" if cl == 1 else "NON_PERSPECTIVE")
                num += 1

    if dict_out['num']:
        weights_df = pd.DataFrame.from_dict(dict_out)
        weights_df.to_csv(output_model_dir / "weights_predictors_validation.csv", sep=';', index=False)
        top_perspective = weights_df[weights_df['class'] == "PERSPECTIVE"].head(3)
        if not top_perspective.empty:
            top_predictors_str = ", ".join([f"{row['predictors']} ({row['weights']})" for _, row in top_perspective.iterrows()])
        
    elapsed_time = time.time() - start_time
    
    # Validation block
    print("[GIS Tracing] Calculation of C-A fractals according to Shevyrev—in validation...")
    MOPM_array_flat = np.sort(df_val_full['PROSPECTIVITY'][(df_val_full['PROSPECTIVITY'] > 1e-5)].to_numpy())
    if len(MOPM_array_flat) > 10:
        log_MOPM = np.log10(MOPM_array_flat)
        step_value = 0.07
        quantity_log_MOMP = np.arange(np.min(log_MOPM), np.max(log_MOPM) + step_value, step_value)
        area_log_array = np.array([np.log10(np.sum(log_MOPM >= i) + 1e-8) for i in quantity_log_MOMP])
        d_logArea = np.diff(area_log_array) / step_value
        d2_logArea = np.diff(d_logArea) / step_value
        start_value = int(0.4 * len(quantity_log_MOMP))
        minima_raw = argrelextrema(d2_logArea[start_value:], np.less)
        if len(minima_raw) > 0 and len(minima_raw[0]) > 0:
            class_boundary_ind = minima_raw[0] + start_value
        else:
            class_boundary_ind = np.array([int(len(quantity_log_MOMP)/2)])
        class_boundary_values = 10**quantity_log_MOMP[class_boundary_ind]
    else:
        class_boundary_values = np.array([0.5])
    
    pd.DataFrame({"Fractal_Threshold_Probability": class_boundary_values}).to_csv(output_model_dir / "fractal_classes_validation.csv", index=False)

    print("[GIS Tracing] Calculation of Yousefi P-A curves during validation...")
    total_ore_pixels = np.sum(df_val_full[y_col] > 0) or 1
    total_area_pixels = len(df_val_full)
    class_boundaries_values = np.arange(0, 1.1, 0.1)
    deposits_percent, area_percent = [], []
    for cl in class_boundaries_values:
        ind_cl = df_val_full['PROSPECTIVITY'] >= cl
        deposits_percent.append(100 * np.sum((df_val_full[y_col] > 0) & ind_cl) / total_ore_pixels)
        area_percent.append((1 - (np.sum(ind_cl) / total_area_pixels)) * 100)
        
    deposits_percent, area_percent = np.array(deposits_percent), np.array(area_percent)
    
    # Calculating the intersection point of P-A curves using an imported inference method
    from mcservices.model_inference_service import intersection
    px, py_val = intersection(class_boundaries_values, deposits_percent, area_percent)

    # Rendering of P-A curves for efficiency balance
    fig_pa, ax_pa = plt.subplots(figsize=(6, 5))
    ax_pa.plot(class_boundaries_values, deposits_percent, 'r-', label='Prediction rate')
    ax_pa.plot(class_boundaries_values, area_percent, 'g-', label='Area')
    ax_pa.text(px-0.05, py_val+5, f'({px:.2f},{py_val:.2f})', bbox=dict(facecolor='white', alpha=0.6))
    ax_pa.plot([px, px], [0, py_val], 'r--')
    ax_pa.plot([0, px], [py_val, py_val], 'r--')
    ax_pa.plot(px, py_val, 'bo')
    ax_pa.set_ylabel('Percentage of known occurrences (%)')
    ax_pa.set_xlabel('MP (probability) value')
    ax_pa.set_title('P-A Validation Efficiency Curves', fontsize=11, fontweight='bold')
    ax_pa.set_ylim([-1, 102])
    ax_pa.legend(loc='lower left')
    ax_pa.grid(True, linestyle=':', alpha=0.6)
    plt.savefig(output_model_dir / "pa_balance_curves_validation.png", dpi=150, bbox_inches='tight')
    plt.close(fig_pa)

    # Generating log files, a PNG map, and a final GIS raster in GeoTIFF format
    print('Generating log files, a PNG map, and a final GIS raster in GeoTIFF format')
    generate_maxent_artifacts(full_preds, df, roc_auc_val, final_params, elapsed_time, output_model_dir, proj_meta)
    
    return {
        "status": "success",
        "model_id": model_uuid,
        "auc": round(roc_auc_val, 4), #  ROC-AUC for the quality assessment
        "roc_auc": round(roc_auc_val, 4), 
        "pa_intersection": f"({px:.2f}, {py_val:.2f})",
        "artifacts_dir": str(output_model_dir.resolve()),
        "download_url_path": f"/outputs/trained_models/{model_uuid}/", 
        "top_predictors": top_predictors_str
    }

def generate_maxent_artifacts(full_preds, df, roc_auc_val, final_params, elapsed_time, output_dir, proj_meta):
    """Internal GIS method: generates text reports (focused on ROC-AUC), PNG graphics, and GeoTIFF files for QGIS"""
    with open(output_dir / "training_report.txt", "w", encoding="utf-8") as f:
        f.write(f"Modeling train report\n")
        f.write(f"Session UUID: {output_dir.name}\n")
        f.write(f"Time elapsed: {elapsed_time:.2f} sec\n")
        f.write(f"Metric ROC-AUC for independent validation set (30%): {roc_auc_val:.4f}\n") 
        f.write(f"Constructor parametes configuration: {json.dumps(final_params, indent=2)}\n")

    ref_xmin, ref_xmax = proj_meta['ref_xmin'], proj_meta['ref_xmax']
    ref_ymin, ref_ymax = proj_meta['ref_ymin'], proj_meta['ref_ymax']
    ref_xres, ref_yres = proj_meta['ref_xres'], proj_meta['ref_yres']
    
    r_min, r_max = df['rowind'].min(), df['rowind'].max()
    c_min, c_max = df['colind'].min(), df['colind'].max()
    mat_rows, mat_cols = int((r_max - r_min) + 1), int((c_max - c_min) + 1)
    
    pred_matrix = np.ones((mat_rows, mat_cols), dtype=np.float32) * -9999.0
    for r, c, val in zip(df['rowind'], df['colind'], full_preds):
        pred_matrix[r - r_min, c - c_min] = val

    plt.figure(figsize=(8, 6))
    masked_mat = np.ma.masked_where(pred_matrix == -9999.0, pred_matrix)
    plt.imshow(masked_mat, cmap='YlOrRd', interpolation='none')
    plt.colorbar(label='MPM')
    plt.title('Mineral prospectivity map (MaxEnt)')
    plt.savefig(output_dir / "maxent_probability_map.png", dpi=200, bbox_inches='tight')
    plt.close()

    aoi_xmin, aoi_ymax = ref_xmin + (c_min * ref_xres), ref_ymax - (r_min * ref_yres)
    out_tif_path = output_dir / "maxent_prediction.tif"
    driver = gdal.GetDriverByName("GTiff")
    ds = driver.Create(str(out_tif_path), mat_cols, mat_rows, 1, gdal.GDT_Float32)
    ds.SetGeoTransform((aoi_xmin, ref_xres, 0, aoi_ymax, 0, -ref_yres))
    srs = osr.SpatialReference()
    srs.ImportFromEPSG(32653)
    ds.SetProjection(srs.ExportToWkt())
    band = ds.GetRasterBand(1)
    band.WriteArray(pred_matrix)
    band.SetNoDataValue(-9999.0)
    ds.FlushCache(); ds = None
    print(f"[GIS Tracing] All artifacts and GeoTIFFs have been saved into the model folder: {output_dir.resolve()}")

In [ ]:
#lets run training
PROCESSING_DIR = ""
runtime_config = {"use_gridsearch": use_grid.value, "params": {}}

execute_model_training(PROCESSING_DIR,runtime_config)

## Model inference snippet

In [ ]:
def get_inference_dataset(processing_dir: str, am: dict, target_aoi: str):
    """
    Performs a spatial intersection of the mineral layers in the `modlayers` table from the `data.db` database, and calculates the geographic coordinates (LAT/LON) for each row in the dataset.
    """
    log_message("Reading raster projection metadata from the modproj database table...")
    
    db_path = Path(processing_dir) / "data.db"
    conn = sqlite3.connect(str(db_path))
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM modproj LIMIT 1")
    proj_meta = dict(cursor.fetchone())
    
    target_col = str(am["target_variable"]).strip("[]'\" ")
    X_cols = list(am["predictors"].keys())
    all_cols = ["id", "rowind", "colind", target_col] + X_cols
    cols_str = ", ".join([f'"{c}"' for c in all_cols])
    
    if target_aoi == "full":
        log_message("Export of the FULL extent of the entire Primorye matrix from the database...")
        query = f'SELECT {cols_str} FROM modlayers'
        df = pd.read_sql(query, conn)
    else:
        log_message(f"Getting region {target_aoi} for aoi.json boundaries...")
        with open(Path(processing_dir) / "aoi.json", 'r', encoding='utf-8') as f:
            aoi = json.load(f)
        zone = aoi[target_aoi]
        
        start_col = round((zone["xmin_meters"] - proj_meta['ref_xmin']) / proj_meta['ref_xres'])
        end_col = round((zone["xmax_meters"] - proj_meta['ref_xmin']) / proj_meta['ref_xres'])
        start_row = round((proj_meta['ref_ymax'] - zone["ymax_meters"]) / proj_meta['ref_yres'])
        end_row = start_row + round((zone["ymax_meters"] - zone["ymin_meters"]) / proj_meta['ref_yres'])
        
        query = f"""
            SELECT {cols_str} FROM modlayers 
            WHERE rowind >= {min(start_row, end_row)} AND rowind <= {max(start_row, end_row)}
              AND colind >= {min(start_col, end_col)} AND colind <= {max(start_col, end_col)}
        """
        df = pd.read_sql(query, conn)
    conn.close()
    
    log_message("Cleaning up NoData (-1.0) values ​​and converting meters to geographic degrees (LAT/LON)...")
    nodata_val = am.get("nodata_replacement_value", 0.0)
    for p in X_cols:
        df[p] = df[p].replace(-1.0, nodata_val)
        
    x_meters = proj_meta['ref_xmin'] + (df['colind'] * proj_meta['ref_xres'])
    y_meters = proj_meta['ref_ymax'] - (df['rowind'] * proj_meta['ref_yres'])
    
    source_srs = osr.SpatialReference()
    source_srs.ImportFromEPSG(32653) # UTM 53N
    target_srs = osr.SpatialReference()
    target_srs.ImportFromEPSG(4326)  # WGS84 degrees
    if hasattr(source_srs, 'SetAxisMappingStrategy'):
        source_srs.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)
    transform = osr.CoordinateTransformation(source_srs, target_srs)
    
    lats, lons = [], []
    for xm, ym in zip(x_meters, y_meters):
        pt = transform.TransformPoint(xm, ym)
        lats.append(pt[0]) # Latitude (Y)
        lons.append(pt[1]) # Longitude (X)
        
    df['LAT'] = lats
    df['LON'] = lons
    log_message(f"Data rows extracted and georeferenced: {len(df)}")
    return df, proj_meta


def generate_pdp_curves(pipeline, df_test, X_cols, output_dir):
    """
    Constructing partial dependence plots (PDPs) based on your canonical polynomial. 
    Uses a second-degree approximation on a 1% sample.
    """
    log_message("Generating partial dependence plots (PDPs) on a 1% sample...")
    try:
        df_test = df_test.reset_index(drop=True)
        X_raw = df_test[X_cols]
        X_scaled_arr = pipeline.named_steps['preprocessor'].transform(X_raw)
        if hasattr(X_scaled_arr, "toarray"): 
            X_scaled_arr = X_scaled_arr.toarray()
        X_scaled_df = pd.DataFrame(X_scaled_arr, columns=X_cols)
        
        sample_sz = max(100, int(0.01 * len(X_scaled_df)))
        idx = np.random.choice(X_scaled_df.index, size=min(sample_sz, len(X_scaled_df)), replace=False)
        
        cc = 4
        rr = int(np.ceil(len(X_cols) / cc))
        fig, axes = plt.subplots(rr, cc, figsize=(16, 4 * rr))
        axes = axes.flatten()
        
        deg, bgrdval = 2, 0.05
        for ind, key in enumerate(X_cols):
            x_val = X_scaled_df.loc[idx, key].to_numpy()
            y_val = df_test.loc[idx, 'PROSPECTIVITY'].to_numpy()
            x_nodgrd = x_val[y_val > bgrdval]
            y_nodgrd = y_val[y_val > bgrdval]
            
            ax = axes[ind]
            ax.plot(x_val, y_val, 'b+', alpha=0.2)
            if len(x_nodgrd) > 5:
                coefficients = np.polyfit(x_nodgrd, y_nodgrd, deg=deg)
                polynomial = np.poly1d(coefficients)
                x_fit = np.linspace(min(x_nodgrd), max(x_nodgrd), 30)
                ax.plot(x_fit, polynomial(x_fit), 'r-', linewidth=2)
                
            
            ax.set_title(key, fontsize=10)
            ax.grid(True, linestyle=':', alpha=0.5)
            
        for j in range(len(X_cols), len(axes)): 
            fig.delaxes(axes[j])
            
        plt.tight_layout()
        plt.savefig(output_dir / "pd_curves_plot.png", dpi=150, bbox_inches='tight')
        plt.savefig(output_dir / "pd_curves_plot.svg", dpi=150, bbox_inches='tight')
        plt.close()
        log_message("The graphical report of the PDP curves has been successfully burned to disk.")
    except Exception as ex:
        log_message(f"PDP rendering warning: {str(ex)}")


def run_model_inference_pipeline(processing_dir: str, target_aoi: str, use_alghofaili: bool = False,msg_queue=None):
    """Final implementation method for model testing and comprehensive forecasting"""
    global INF_LOGS,_CURRENT_MSG_QUEUE
    _CURRENT_MSG_QUEUE = msg_queue #message queue
    
    try:
        log_message("Model test for AOI...")
        if isinstance(target_aoi, list) and len(target_aoi) > 0: 
            target_aoi = target_aoi[0]
        target_aoi = str(target_aoi).strip("[]'\" ")
        
        with open("active_model.json", "r", encoding="utf-8") as f:
            am = json.load(f)
            
        model_uuid = str(am["model_id"]).strip("[]'\" ")
        model_filename = str(am["model_filename"]).strip("[]'\" ")
        str_proc_dir = str(processing_dir).strip("[]'\" ")
        
        # Path joining
        pkl_path = Path(os.path.join(str_proc_dir, "trained_models", model_uuid, model_filename))
        
        if not pkl_path.exists():
            return {"status": "error", "message": f"Model file {model_filename} is not found here: {pkl_path}"}
            
        log_message(f"Load model dump file {model_filename} from hard drive...")
        with open(pkl_path, 'rb') as f:
            pipeline = pickle.load(f)
            
        # 1. Loading dataset
        df, proj_meta = get_inference_dataset(str_proc_dir, am, target_aoi)
        X_cols = list(am["predictors"].keys())
        target_col = str(am["target_variable"]).strip("[]'\" ")
        X = df[X_cols]
        
        # 2. Model inference OneClassSVM / MLP
        log_message("Data processing with model MaxEnt OneClassSVM...")
        X = df[X_cols]
        
        # # Getting density logarithms for OneClassSVM (doesn't work with MLP) 
        # raw_scores = pipeline.score_samples(X)

        # Getting estimator type
        final_estimator = pipeline.named_steps[list(pipeline.named_steps.keys())[-1]]
        estimator_name = final_estimator.__class__.__name__
        
        log_message(f"Core architecture: {estimator_name}")
        
        # Dynamic selection of an inference method tailored to the model
        if hasattr(pipeline, 'score_samples'):
            # Variant for OneClassSVM / IsolationForest
            raw_scores = pipeline.score_samples(X)
        elif hasattr(pipeline, 'predict_proba'):
            # Variant for MLPClassifier / RandomForestClassifier (target class probability is 1)
            # predict_proba returns [0 .. 1]
            raw_scores = pipeline.predict_proba(X)[:, 1]
        elif hasattr(pipeline, 'predict'):
            # Variant for MLPRegressor / LinearRegression
            raw_scores = pipeline.predict(X)
        else:
            raise AttributeError(f"Loaded model {estimator_name} doesn't support any predictivity method")
            

        min_s, max_s = raw_scores.min(), raw_scores.max()
        
        # We normalize this into a neat GIS index of environmental similarity, ranging from 0.0 to 1.0.
        df['PROSPECTIVITY'] = (raw_scores - min_s) / (max_s - min_s + 1e-8)

        # Export the extended validation table to hard disk drive
        probation_path_str = os.path.join(str_proc_dir, "tested_models", model_uuid, target_aoi, "approbation")
        probation_dir = Path(probation_path_str)
        probation_dir.mkdir(parents=True, exist_ok=True)
        df.to_csv(probation_dir / "approbation_data.csv", index=False)

        # 3. Validation P-A и ROC-AUC computations
        log_message("Computing ROC-AUC and fractal C-A class boundary analysis...")
        y_true = (df[target_col] > 0).astype(int)
        scores = df['PROSPECTIVITY'].to_numpy()
        
        # Compute ROC-AUC for inference matrix
        fpr, tpr, _ = roc_curve(y_true, scores)
        roc_auc = sklearn_auc(fpr, tpr)
        pd.DataFrame({"FPR": fpr, "TPR": tpr}).to_csv(probation_dir / "validation_metrics.csv", index=False)

        log_message("Constructing and rendering a canonical ROC curve for GIS validation...")
        fig_roc, ax_roc = plt.subplots(figsize=(6, 5)) 
        
        # Compute MaxEnt curve
        ax_roc.plot(fpr, tpr, color='darkorange', linewidth=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
        
        # Compute Random Guess line
        ax_roc.plot([0, 1], [0, 1], color='navy', linewidth=2, linestyle='--', label='Random Guess')
        
        # We set fixed axis limits 
        ax_roc.set_xlim([0.0, 1.0])
        ax_roc.set_ylim([0.0, 1.05])
        
        # Axis labels and title interface
        ax_roc.set_xlabel('False Positive Rate (FPR)', fontsize=10)
        ax_roc.set_ylabel('True Positive Rate (TPR)', fontsize=10)
        ax_roc.set_title(f'Receiver Operating Characteristic (ROC) - {target_aoi}', fontsize=11, fontweight='bold')
        
        # Legend and grid
        ax_roc.legend(loc="lower right", fontsize=9)
        ax_roc.grid(True, linestyle=':', alpha=0.6, color='gray')
        
        # Saving a ROC graphical report to the file system securely
        try:
            plt.savefig(probation_dir / "roc_curve_validation.png", dpi=150, bbox_inches='tight')
            plt.savefig(probation_dir / "roc_curve_validation.svg", dpi=150, bbox_inches='tight')
            log_message("The ROC curve graphical report has been successfully saved to disk.")
        except Exception as e_roc_save:
            log_message(f"ROC-AUC plotting warning: {e_roc_save}")
        finally:
            plt.close(fig_roc) #Close figure 

        # C-A fractal curves
        MOPM_array_flat = np.sort(df['PROSPECTIVITY'][(df['PROSPECTIVITY'] > 1e-5)].to_numpy())
        log_MOPM = np.log10(MOPM_array_flat)
        step_value = 0.07
        quantity_log_MOMP = np.arange(np.min(log_MOPM), np.max(log_MOPM) + step_value, step_value)
        area_log_array = np.array([np.log10(np.sum(log_MOPM >= i) + 1e-8) for i in quantity_log_MOMP])
        d_logArea = np.diff(area_log_array) / step_value
        d2_logArea = np.diff(d_logArea) / step_value
        start_value = int(0.4 * len(quantity_log_MOMP))
        minima_raw = argrelextrema(d2_logArea[start_value:], np.less)
        
        # Getting class minimal value
        if len(minima_raw) > 0 and len(minima_raw[0]) > 0:
            class_boundary_ind = minima_raw[0] + start_value
        else:
            class_boundary_ind = np.array([int(len(quantity_log_MOMP)/2)])
            
        class_boundary_values = 10**quantity_log_MOMP[class_boundary_ind]
        pd.DataFrame({"Fractal_Threshold_Probability": class_boundary_values}).to_csv(probation_dir / "fractal_classes.csv", index=False)

        # Yousefi & Carranza (2015) P-A efficiency curves
        log_message("Construction of P-A curves for balancing forecast performance")
        total_ore_pixels = np.sum(df[target_col] > 0) or 1
        total_area_pixels = len(df)
        class_boundaries_values = np.arange(0, 1.1, 0.1)
        deposits_percent, area_percent = [], []
        for cl in class_boundaries_values:
            ind = df['PROSPECTIVITY'] >= cl
            deposits_percent.append(100 * np.sum((df[target_col] > 0) & ind) / total_ore_pixels)
            area_percent.append((1 - (np.sum(ind) / total_area_pixels)) * 100)
            
        deposits_percent, area_percent = np.array(deposits_percent), np.array(area_percent)

        log_message("Compose P-A efficiency balance curves...")
        total_ore_pixels = np.sum(df[target_col] > 0) or 1
        total_area_pixels = len(df)
        class_boundaries_values = np.arange(0, 1.1, 0.1)
        
        deposits_percent, area_percent = [], []
        for cl in class_boundaries_values:
            ind = df['PROSPECTIVITY'] >= cl
            deposits_percent.append(100 * np.sum((df[target_col] > 0) & ind) / total_ore_pixels)
            area_percent.append((1 - (np.sum(ind) / total_area_pixels)) * 100)
            
        deposits_percent, area_percent = np.array(deposits_percent), np.array(area_percent)
        
        # Getting intersection value
        px, py_val = intersection(class_boundaries_values, deposits_percent, area_percent)

        fig, ax1 = plt.subplots(figsize=(6, 5))
        ax1.plot(class_boundaries_values, deposits_percent, 'r-', label='Prediction rate')
        ax1.plot(class_boundaries_values, area_percent, 'g-', label='Area')
        
        # Plot the equilibrium point on the P-A graph
        ax1.text(px-0.05, py_val+5, f'({px:.2f},{py_val:.2f})', bbox=dict(facecolor='white', alpha=0.6))
        ax1.plot([px, px], [0, py_val], 'r--')
        ax1.plot([0, px], [py_val, py_val], 'r--')
        ax1.plot(px, py_val, 'bo')
        
        ax1.set_ylabel('Percentage of known occurrences (%)')
        ax1.set_xlabel('MP (probability) value')
        ax1.set_ylim([-1, 102])
        ax1.legend(loc='lower left')
        ax1.grid(True)

        #Saving the P-A curve of the balance of efficiency on the disk
        try:
            plt.savefig(probation_dir / "pa_balance_curves.png", dpi=150, bbox_inches='tight')
            plt.savefig(probation_dir / "pa_balance_curves.svg", dpi=150, bbox_inches='tight')
            log_message("The graphical report of the Yousefi P-A curves has been successfully saved to disk")
        except Exception as e_save:
            log_message(f"Warning when saving P-A plots: {e_save}")
        finally:
            plt.close(fig) # Memory offload


        # Alghofaili method implementation:
        if use_alghofaili:
            try:
                log_message(f"The Alghofaili block for feature importance assessment. Training of the RandomForestClassifier will begin at the boundary  {px:.2f}...")
                pixel_classes = np.zeros(len(df), dtype=int)
                pixel_classes[df['PROSPECTIVITY'] >= px] = 1
                dict_out = {'num': [], 'predictors': [], 'weights': [], 'class': []}
                num = 0
                
                # Iteration over two classes (Foreground and Background)
                for cl in [0,1]:
                    clf = RandomForestClassifier(n_estimators=50, max_depth=8, n_jobs=1, random_state=1)
                    series = np.zeros(len(X), dtype=int)
                    series[pixel_classes == cl] = 1
                    clf.fit(X.values, series)
                    
                    for ind in np.argsort(clf.feature_importances_)[::-1]:
                        dict_out['num'].append(num)
                        dict_out['predictors'].append(X_cols[ind])
                        dict_out['weights'].append(round(float(clf.feature_importances_[ind]), 4))
                        dict_out['class'].append("PERSPECTIVE" if cl == 1 else "NON_PERSPECTIVE")
                        num += 1
                pd.DataFrame.from_dict(dict_out).to_csv(probation_dir / "weights_predictors.csv", sep=';', index=False)
                log_message("The importance analysis according to Algofaili is complete, and the CSV file containing the weights has been saved.")
            except Exception as e_rf: 
                log_message(f"Alghofaili computation error: {e_rf}")

        # PDP curve creating
        generate_pdp_curves(pipeline, df, X_cols, probation_dir)

        # Saving a GeoTIFF raster using GDAL
        log_message("Writing a georeferenced GeoTIFF raster using GDAL C++ drivers...")
        r_min, r_max = df['rowind'].min(), df['rowind'].max()
        c_min, c_max = df['colind'].min(), df['colind'].max()
        
        # Cast dimensions strictly to `int` for the C-based `GDAL.Driver_Create` methods.
        mat_rows = int((r_max - r_min) + 1)
        mat_cols = int((c_max - c_min) + 1)
        
        pred_matrix = np.ones((mat_rows, mat_cols), dtype=np.float32) * -9999.0
        for r, c, val in zip(df['rowind'], df['colind'], df['PROSPECTIVITY']):
            pred_matrix[r - r_min, c - c_min] = val
            
        plt.figure(figsize=(7, 6))
        masked_mat = np.ma.masked_where(pred_matrix == -9999.0, pred_matrix)
        plt.imshow(masked_mat, cmap='YlOrRd', interpolation='none')
        plt.colorbar(label='MaxEnt Probability')
        plt.savefig(probation_dir / "prospectivity_map.png", dpi=150, bbox_inches='tight')
        plt.savefig(probation_dir / "prospectivity_map.svg", dpi=150, bbox_inches='tight')
        plt.close()

        aoi_xmin = proj_meta['ref_xmin'] + (c_min * proj_meta['ref_xres'])
        aoi_ymax = proj_meta['ref_ymax'] - (r_min * proj_meta['ref_yres'])
        
        out_tif_path = probation_dir / "prospectivity_prediction.tif"
        driver = gdal.GetDriverByName("GTiff")
        ds = driver.Create(str(out_tif_path), mat_cols, mat_rows, 1, gdal.GDT_Float32)
        ds.SetGeoTransform((aoi_xmin, proj_meta['ref_xres'], 0, aoi_ymax, 0, -proj_meta['ref_yres']))
        srs = osr.SpatialReference()
        srs.ImportFromEPSG(32653)
        ds.SetProjection(srs.ExportToWkt())
        band = ds.GetRasterBand(1)
        band.WriteArray(pred_matrix)
        band.SetNoDataValue(-9999.0)
        ds.FlushCache(); ds = None

        log_message("The validation pipeline has successfully completed the calculation")
        return {
            "status": "success",
            "roc_auc": round(roc_auc, 4),
            "pa_intersection": f"({px:.2f}, {py_val:.2f})", # We’re passing raw float values directly
            "download_url_path": f"/outputs/tested_models/{model_uuid}/{target_aoi}/approbation/"
        }
    except Exception as e:
        log_message(f"Pipeline error: {str(e)}")
        return {"status": "error", "message": str(e)}

In [ ]:
#run inference pipeline
PROCESSING_DIR=""
target_aoi="aoi"
run_model_inference_pipeline(processing_dir, target_aoi)

## Modeling results assessing with logarithmic C-A curves

References:

   
Yousefi, M., Carranza, E.J.M., 2015. Prediction area (P-A) plot and C-A fractal analysis to classify and maps for mineral prospectivity modelling. Comput. Geosci. 79, 69–81.

Sergei Shevyrev, Carranza Em. J. M.  Application of maximum entropy for mineral prospectivity mapping in heavily vegetated areas of Greater Kurile Chain with Landsat 8 data, Ore Geology Reviews, Volume 142, 2022, 104758 https://doi.org/10.1016/j.oregeorev.2022.104758.


In [ ]:

from osgeo import gdal, ogr
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import time
import pickle
from scipy.signal import argrelextrema

files_path = {
    "MaxEnt model prediction":os.path.join('..','..','files','trained_models','model_folder','prediction.tif'),
}

for filename,filepath in files_path.items():
    
    # 1. Adjustment
    predicted_file_path = filepath #tiff name for opening
    class_boundaries_value_filename = f'cbv{filename}.values'
    
    filter_extrema = 0.1 # First derivative threshold
    start_value_perc = 0.5  # Start for data analysis (0.5 median)

    # 2. Opening inference map
    if not os.path.exists(predicted_file_path):
        raise FileNotFoundError(f"Inference filw was not found: {predicted_file_path}")

    MOPM_gdal_object = gdal.Open(predicted_file_path)
    MOPM_array = MOPM_gdal_object.GetRasterBand(1).ReadAsArray()

    # NaN and NoData filtering (-9999)
    MOPM_array[np.isnan(MOPM_array)] = 0
    # Loading pixels from AOI
    MOPM_array_flat = np.ndarray.flatten(MOPM_array[(MOPM_array > 1e-5) & (MOPM_array != -9999)])
    MOPM_array_flat.sort()

    # 3. Compute logarithmic curves C-A
    log_MOPM = np.log10(MOPM_array_flat)
    data_range = np.max(log_MOPM) - np.min(log_MOPM)
    step_value = data_range / 200
    if step_value < 0.005: step_value = 0.005 

    # construct a grid of log-probabilities.
    quantity_log_MOMP = np.arange(np.min(log_MOPM), np.max(log_MOPM) + step_value, step_value)
    start_value = int(start_value_perc * len(quantity_log_MOMP))

    area_log_array = np.array([])
    for i in quantity_log_MOMP:
        # Cumulative area: the number of pixels with a probability greater than or equal to a given value
        count = np.sum(log_MOPM >= i)
        if count > 0:
            area_log_array = np.append(area_log_array, np.log10(count))
        else:
            area_log_array = np.append(area_log_array, 0) # Protection log10(0) -> -inf

    # First derivative: d_logArea / d_logMOPM
    d_logArea = np.diff(area_log_array) / step_value
    quantity_log_MOMP_d1 = quantity_log_MOMP[:-1]

    # Second derivative
    d2_logArea = np.diff(d_logArea) / step_value
    quantity_log_MOMP_d2 = quantity_log_MOMP[:-2]

    # Convert directly into 1D arrays of indices
    minima_raw = argrelextrema(d2_logArea[start_value:], np.less)[0]
    maxima_raw = argrelextrema(d2_logArea[start_value:], np.greater)[0]

    # Shift the indices back to the global scale relative to start_value!
    minima = minima_raw + start_value
    maxima = maxima_raw + start_value

    # The indices correspond strictly to the d_logArea array.
    minima_filtered = minima[np.abs(d_logArea[minima]) > filter_extrema]
    maxima_filtered = maxima[np.abs(d_logArea[maxima]) > filter_extrema]

    # Combine and sort the indices of the final class boundaries
    class_boundary_ind = np.unique(np.append(minima_filtered, maxima_filtered))
    class_boundary_ind.sort()

    # Combine closest boundaries
    raw_log_values = quantity_log_MOMP[class_boundary_ind]
    min_distance_delta = 0.05  # If coundaries distance less than 0.05 — join them!

    merged_log_values = []
    if len(raw_log_values) > 0:
        current_cluster = [raw_log_values[0]]
        for val in raw_log_values[1:]:
            if val - current_cluster[-1] < min_distance_delta:
                current_cluster.append(val)
            else:
                merged_log_values.append(np.mean(current_cluster))
                current_cluster = [val]
        merged_log_values.append(np.mean(current_cluster))

    class_boundary_values_log = np.array(merged_log_values)

    #Plot exporting
    # Getting correct indices for joint borders
    final_class_boundary_ind = []
    for val in class_boundary_values_log:
        idx = (np.abs(quantity_log_MOMP - val)).argmin()
        final_class_boundary_ind.append(idx)
    final_class_boundary_ind = np.unique(final_class_boundary_ind)

    # Plot 1: Основной Fractal C-A 
    plt.figure(figsize=(10, 6))
    plt.plot(quantity_log_MOMP[start_value:], area_log_array[start_value:], 'b-', label='C-A Curve', linewidth=2)
    plt.title(f'{filename} fractal C-A Plot', fontsize=12)
    plt.xlabel('Log10(Mineral Prospectivity)')
    plt.ylabel('Log10(Cumulative Area)')

    # Draw only noise-free inflection points
    plt.plot(quantity_log_MOMP[final_class_boundary_ind], area_log_array[final_class_boundary_ind], "xr", markersize=10, label='Merged Inflection Points')

    # Draw vertical hatch lines
    y_min_val = np.min(area_log_array[start_value:])
    for ind in final_class_boundary_ind:
        x_val = quantity_log_MOMP[ind]
        y_val = area_log_array[ind]
        plt.plot([x_val, x_val], [y_min_val, y_val], 'r--')

    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='lower left')
    plt.savefig(f'{filename}C-A_plot.png', dpi=300)
    plt.savefig(f'{filename}C-A_plot.svg', dpi=300)
    plt.close()

    # Plot 2: first derivative
    plt.figure(figsize=(10, 4))
    plt.plot(quantity_log_MOMP_d1[start_value:], d_logArea[start_value:], 'g-', label='1st Derivative')
    plt.title('First Derivative of C-A Curve')
    plt.xlabel('Log10(MOPM Value)')
    plt.ylabel('d(Log Area) / d(Log MOPM)')
    plt.grid(True, linestyle=':')
    plt.legend()
    plt.close()

    # Plot 3: second derivative
    plt.figure(figsize=(10, 4))
    plt.plot(quantity_log_MOMP_d2[start_value:], d2_logArea[start_value:], 'm-', label='2nd Derivative')
    plt.title('Second Derivative of C-A Curve (Curvature)')
    plt.xlabel('Log10(MOPM Value)')
    plt.ylabel('d2(Log Area) / d2(Log MOPM)')
    plt.grid(True, linestyle=':')
    plt.legend()
    plt.close()

    # Exporting prospectivity class boundaries 
    # Converting from lorarithm scale into normal (0...1)
    class_boundary_values = 10**class_boundary_values_log

    print('\nClass boundaries were found')
    print(f"Scale (Log10): {class_boundary_values_log}")
    print(f"Normal scale OneClassSVM (0-1): {class_boundary_values}")

    with open(class_boundaries_value_filename, "w", encoding="utf-8") as file:
        for item in class_boundary_values:
            file.write(f"{item}\n")

    print(f"Boundary values have been exported: {class_boundaries_value_filename}")
